In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
.appName("PySparkPractice") \
.getOrCreate()

In [3]:
products_data = [
("P01", "Laptop", "Electronics", 60000),
("P02", "Monitor", "Electronics", 14000),
("P03", "Headphones", "Accessories", 3500),
("P04", "Printer", "Office", 17000),
("P05", "Router", "Networking", 4500),
("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
products_data,
["product_id", "product_name", "category", "list_price"]
)
stores_data = [
("S01", "Hyderabad", "South"),
("S02", "Bengaluru", "South"),
("S03", "Mumbai", "West"),
("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
stores_data, ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()

+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+



In [4]:
sales_base_df = (
spark.range(1, 100001)
.withColumnRenamed("id", "order_id")
.withColumn(
"order_date",
F.date_add(
F.to_date(F.lit("2026-01-01")),
(F.col("order_id") % 180).cast("int")
)
)
.withColumn(
"product_id",
F.concat(
F.lit("P"),
F.lpad(
((F.col("order_id") % 6) + 1).cast("string"),
2, "0"
)
)
)
.withColumn(
"store_id",
F.concat(
F.lit("S"),
F.lpad(
((F.floor(F.col("order_id") / 3) % 4) + 1)
.cast("string"),
2, "0"
)
)
)
.withColumn(
"quantity",
((F.col("order_id") % 8) + 1).cast("int")
)
.withColumn(
"discount_pct",
F.when(
F.col("order_id") % 19 == 0,
F.lit(None).cast("int")
).otherwise(
((F.floor(F.col("order_id") / 5) % 4) * 5)
.cast("int")
)
)
)

In [5]:
sales_df = (
sales_base_df
.withColumn(
"order_status",
F.when(F.col("order_id") % 11 == 0, "Cancelled")
.when(F.col("order_id") % 7 == 0, "Returned")
.otherwise("Completed")
)
.withColumn(
"payment_method",
F.when(F.col("order_id") % 3 == 0, "UPI")
.when(F.col("order_id") % 3 == 1, "Card")
.otherwise("Cash")
)
.withColumn(
"salesperson_id",
F.when(
F.col("order_id") % 17 == 0,
F.lit(None).cast("string")
).otherwise(
F.concat(
F.lit("EMP"),
F.lpad(
((F.col("order_id") % 12) + 1).cast("string"),
2, "0"
)
)
)
)
.join(
products_df.select(
"product_id",
F.col("list_price").alias("unit_price")
),
"product_id", "left"
)
.select(
"order_id", "order_date", "store_id",
"product_id", "quantity", "unit_price",
"discount_pct", "order_status",
"payment_method", "salesperson_id"
)
)
print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_status: string (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- salesperson_id: string (nullable = true)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|3       |2026-01-04|S02     |P04       |4       |17000     |0           |Completed   |UPI           |EMP04         |
|9       |2026-01-10|S04     |P04       |2       |17000     |5           |C

In [6]:
df=sales_df.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
).show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|      15|       P04|       8|     17000|
|      21|       P04|       6|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|      14|       P03|       7|      3500|
|      20|       P03|       5|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|      16|       P05|       1|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|      17|       P06|       2|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
|      13|       P02|       6|     14000|
|      19|       P02|       4|     14000|
|       6|       P01|       7|     60000|
|      12|       P01|       5|     60000|
+--------+----------+--------+----

In [8]:
from pyspark.sql.functions import col

df = sales_df.filter(
    (col("order_status") == "Completed") &
    (col("quantity") > 5)
)
df.show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      69|2026-03-11|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|      62|2026-03-04|     S01|       P03|       7|      

In [9]:
df=sales_df.withColumn(
    "gross_value",
    col("quantity") * col("unit_price")
).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-----------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_value|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-----------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|      68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|      34000|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|     136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|     102000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|   

In [10]:
df=sales_df.withColumnRenamed(
    "order_status",
    "transaction_status"
).drop("payment_method").show()

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|          Returned|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|         Completed|         EMP09|
|      14|2026-01-1

In [12]:
df=sales_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).groupBy(
    "product_id"
).agg(
    F.sum("gross_amount").alias("total_sales")
).show()

+----------+-----------+
|product_id|total_sales|
+----------+-----------+
|       P04| 1416678000|
|       P03|  233334500|
|       P05|  299992500|
|       P06| 1999920000|
|       P02| 1166704000|
|       P01| 4000080000|
+----------+-----------+



In [13]:
from pyspark.sql.functions import sum
result_df=sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).groupBy("category").agg(
    sum(
        col("quantity") * col("unit_price")
    ).alias("total_sales")
    ).show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|     Office| 1416678000|
| Networking|  299992500|
|Electronics| 7166704000|
|Accessories|  233334500|
+-----------+-----------+



In [14]:
result_df=sales_df.join(
    stores_df,
    on="store_id",
    how="inner"
).groupBy("city").agg(
    sum(
        col("quantity") * col("unit_price")
    ).alias("total_sales")
    ).show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|   Mumbai| 3287509500|
|    Delhi| 1953997500|
|Bengaluru| 1762593000|
|Hyderabad| 2112609000|
+---------+-----------+



In [17]:
sales_df=sales_df.withColumn(
    "sales_amount",
    col("quantity") * col("unit_price")
)

joined_df=sales_df \
.join(products_df, on="product_id", how="inner") \
.join(stores_df, on="store_id", how="inner")

result=joined_df.filter(
    col("order_status")=="Completed"
).groupBy("product_name").agg(
    sum("sales_amount").alias("total_sales")
).orderBy(
    "total_sales", ascending=False
).show(5)

+------------+-----------+
|product_name|total_sales|
+------------+-----------+
|      Laptop| 3116940000|
|      Tablet| 1558416000|
|     Printer| 1103912000|
|     Monitor|  909076000|
|      Router|  233766000|
+------------+-----------+
only showing top 5 rows


In [21]:
df2 = sales_df.filter(F.col("discount_pct").isNull())
print("Number of nulls:", df2.count())
df2=sales_df.filter(F.col("salesperson_id").isNull())
print("Number of nulls:", df2.count())


Number of nulls: 5263
Number of nulls: 5882


In [22]:
df=sales_df.na.fill({
    "discount_pct":0}
).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|sales_amount|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|       68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|       34000|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|      136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|      102000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Comple

In [23]:
df=sales_df.na.drop(
    subset=["salesperson_id"]
).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|sales_amount|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|       68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|       34000|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|      136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|      102000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Comple

In [24]:
from pyspark.sql.functions import when, col

df = sales_df.withColumn(
    "order_size",
    when(col("quantity").between(1, 3), "Small")
    .when(col("quantity").between(4, 6), "Medium")
    .otherwise("Large")
)

df.select(
    "quantity",
    "order_size"
).show()

+--------+----------+
|quantity|order_size|
+--------+----------+
|       4|    Medium|
|       2|     Small|
|       8|     Large|
|       6|    Medium|
|       3|     Small|
|       1|     Small|
|       7|     Large|
|       5|    Medium|
|       5|    Medium|
|       3|     Small|
|       1|     Small|
|       6|    Medium|
|       4|    Medium|
|       2|     Small|
|       2|     Small|
|       8|     Large|
|       6|    Medium|
|       4|    Medium|
|       7|     Large|
|       5|    Medium|
+--------+----------+
only showing top 20 rows


In [28]:
from pyspark.sql.functions import sum, col
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

sales_with_products = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
)

product_revenue = sales_with_products.groupBy(
    "category",
    "product_name"
).agg(
    sum(
        col("quantity") * col("unit_price")
    ).alias("total_gross_revenue")
)

w = Window.partitionBy("category") \
          .orderBy(col("total_gross_revenue").desc())

result = product_revenue.withColumn(
    "rank",
    dense_rank().over(w)
)

result.show()

+-----------+------------+-------------------+----+
|   category|product_name|total_gross_revenue|rank|
+-----------+------------+-------------------+----+
|Accessories|  Headphones|          233334500|   1|
|Electronics|      Laptop|         4000080000|   1|
|Electronics|      Tablet|         1999920000|   2|
|Electronics|     Monitor|         1166704000|   3|
| Networking|      Router|          299992500|   1|
|     Office|     Printer|         1416678000|   1|
+-----------+------------+-------------------+----+



In [30]:
w = Window.partitionBy("store_id") \
          .orderBy("order_date", "order_id")

# Create gross_amount first, then calculate the running total
result = sales_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).withColumn(
    "running_gross_revenue",
    sum(col("gross_amount")).over(w)
)

result.select(
    "store_id",
    "order_date",
    "order_id",
    "gross_amount",
    "running_gross_revenue"
).show()

+--------+----------+--------+------------+---------------------+
|store_id|order_date|order_id|gross_amount|running_gross_revenue|
+--------+----------+--------+------------+---------------------+
|     S01|2026-01-01|     180|      300000|               300000|
|     S01|2026-01-01|     360|       60000|               360000|
|     S01|2026-01-01|     540|      300000|               660000|
|     S01|2026-01-01|     720|       60000|               720000|
|     S01|2026-01-01|     900|      300000|              1020000|
|     S01|2026-01-01|    1080|       60000|              1080000|
|     S01|2026-01-01|    1260|      300000|              1380000|
|     S01|2026-01-01|    1440|       60000|              1440000|
|     S01|2026-01-01|    1620|      300000|              1740000|
|     S01|2026-01-01|    1800|       60000|              1800000|
|     S01|2026-01-01|    1980|      300000|              2100000|
|     S01|2026-01-01|    2160|       60000|              2160000|
|     S01|

In [31]:
from pyspark.sql.functions import month, sum, col

result = sales_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).withColumn(
    "month",
    month("order_date")
).groupBy(
    "month"
).agg(
    sum("gross_amount").alias("monthly_sales_revenue")
).orderBy(
    "month"
)

result.show()

+-----+---------------------+
|month|monthly_sales_revenue|
+-----+---------------------+
|    1|           1278728500|
|    2|           1027585500|
|    3|           1212087500|
|    4|           1218921000|
|    5|           1277560500|
|    6|           1089027500|
+-----+---------------------+



In [32]:
from pyspark.sql.functions import col

# Transformation 1: filter()
df1 = sales_df.filter(col("quantity") > 2)
print("Filter transformation created")

# Transformation 2: withColumn()
df2 = df1.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)
print("withColumn transformation created")

# Transformation 3: select()
df3 = df2.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "gross_amount"
)
print("Select transformation created")

# Action (execution triggered here)
df3.show()

Filter transformation created
withColumn transformation created
Select transformation created
+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
|       3|       P04|       4|     17000|       68000|
|      15|       P04|       8|     17000|      136000|
|      21|       P04|       6|     17000|      102000|
|      27|       P04|       4|     17000|       68000|
|       2|       P03|       3|      3500|       10500|
|      14|       P03|       7|      3500|       24500|
|      20|       P03|       5|      3500|       17500|
|      26|       P03|       3|      3500|       10500|
|       4|       P05|       5|      4500|       22500|
|      10|       P05|       3|      4500|       13500|
|      22|       P05|       7|      4500|       31500|
|      28|       P05|       5|      4500|       22500|
|       5|       P06|       6|     24000|      144000|
|      11|       P06|     

In [33]:
from pyspark.sql.functions import sum

product_sales = sales_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).groupBy("product_id").agg(
    sum("gross_amount").alias("total_gross_revenue")
)

# View execution plan
product_sales.explain()

# Trigger execution
result = product_sales.collect()

for row in result[:10]:
    print(row)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[sum(gross_amount#655L)])
   +- HashAggregate(keys=[product_id#33], functions=[partial_sum(gross_amount#655L)])
      +- Project [product_id#33, (cast(quantity#35 as bigint) * unit_price#40L) AS gross_amount#655L]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=5235]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=5236]
                  +- Project [product_id#0, list_price#3L AS unit_price#40

In [34]:
import time
from pyspark.sql.functions import col

# Filter completed orders
completed_df = sales_df.filter(
    col("order_status") == "Completed"
)

# Cache DataFrame
completed_df.cache()

# First count()
start = time.time()
count1 = completed_df.count()
end = time.time()

print("First Count:", count1)
print("First Execution Time:", end - start, "seconds")

# Second count()
start = time.time()
count2 = completed_df.count()
end = time.time()

print("Second Count:", count2)
print("Second Execution Time:", end - start, "seconds")


First Count: 77923
First Execution Time: 6.012409687042236 seconds
Second Count: 77923
Second Execution Time: 1.7342166900634766 seconds


In [35]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.functions import col

# Filter records
qty_df = sales_df.filter(
    col("quantity") > 5
)

# Persist
qty_df.persist(StorageLevel.MEMORY_AND_DISK)

# Action 1
print("Record Count:")
print(qty_df.count())

# Action 2
print("Top Records:")
qty_df.show(10)

# Remove persisted data
qty_df.unpersist()

print("Persistence removed")

Record Count:
37500
Top Records:
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|sales_amount|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|      136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|      102000|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|      136000|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|      102000|
|      63|2026-03-05|     S02|       P04|       8|

In [37]:
from pyspark.sql.functions import sum

# Aggregate
agg_df = sales_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).groupBy("product_id").agg(
    sum("gross_amount").alias("total_revenue")
)

# Cache aggregation
agg_df.cache()

# Trigger caching
agg_df.count()

# Inspect execution plan
agg_df.explain(True)

== Parsed Logical Plan ==
'Aggregate ['product_id], ['product_id, 'sum('gross_amount) AS total_revenue#1919]
+- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, sales_amount#301L, (cast(quantity#35 as bigint) * unit_price#40L) AS gross_amount#1918L]
   +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, (cast(quantity#35 as bigint) * unit_price#40L) AS sales_amount#301L]
      +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, (cast(quantity#35 as bigint) * unit_price#40L) AS sales_amount#296L]
         +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, paym

In [38]:
# Current partition count
print("Current Partitions:",
      sales_df.rdd.getNumPartitions())

# Repartition to 8
sales_df_8 = sales_df.repartition(8)

print("New Partitions:",
      sales_df_8.rdd.getNumPartitions())

Current Partitions: 2
New Partitions: 8


In [39]:
from pyspark.sql.functions import spark_partition_id

sales_df_8.withColumn(
    "partition_id",
    spark_partition_id()
).groupBy(
    "partition_id"
).count().orderBy(
    "partition_id"
).show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|12500|
|           1|12500|
|           2|12500|
|           3|12500|
|           4|12500|
|           5|12500|
|           6|12500|
|           7|12500|
+------------+-----+



In [40]:
from pyspark.sql.functions import spark_partition_id

store_df = sales_df.repartition(
    6,
    "store_id"
)

store_df = store_df.withColumn(
    "partition_id",
    spark_partition_id()
)

store_df.select(
    "partition_id",
    "store_id"
).distinct().orderBy(
    "partition_id"
).show(100, truncate=False)

+------------+--------+
|partition_id|store_id|
+------------+--------+
|1           |S03     |
|3           |S02     |
|4           |S01     |
|5           |S04     |
+------------+--------+



In [41]:
df12 = sales_df.repartition(12)

print(df12.rdd.getNumPartitions())

12


In [42]:
coalesce_df = df12.coalesce(4)

print(coalesce_df.rdd.getNumPartitions())

coalesce_df.explain(True)

4
== Parsed Logical Plan ==
Repartition 4, false
+- Repartition 12, true
   +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, (cast(quantity#35 as bigint) * unit_price#40L) AS sales_amount#301L]
      +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, (cast(quantity#35 as bigint) * unit_price#40L) AS sales_amount#296L]
         +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, (cast(quantity#35 as bigint) * unit_price#40L) AS sales_amount#291L]
            +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
          

In [43]:
repartition_df = df12.repartition(4)

print(repartition_df.rdd.getNumPartitions())

repartition_df.explain(True)


4
== Parsed Logical Plan ==
Repartition 4, true
+- Repartition 12, true
   +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, (cast(quantity#35 as bigint) * unit_price#40L) AS sales_amount#301L]
      +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, (cast(quantity#35 as bigint) * unit_price#40L) AS sales_amount#296L]
         +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, (cast(quantity#35 as bigint) * unit_price#40L) AS sales_amount#291L]
            +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
           

In [44]:
sales_df.repartition(4) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("sales_output")


In [47]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    when
)
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

#1
completed_df = sales_df.filter(
    col("order_status") == "Completed"
)

#2
completed_df=completed_df.na.fill(
    {
        "discount_pct": 0
    }
)

#3
joined_df=completed_df \
.join(products_df, on="product_id", how="inner") \
.join(stores_df, on="store_id", how="inner")

#4
revenue_df=joined_df \
.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).withColumn(
        "discounted_revenue",
        col("gross_amount") *
        (1 - col("discount_pct") / 100)
    )

#5,#6
agg_df=revenue_df.groupBy(
    "region",
    "category"
).agg(
    sum("discounted_revenue").alias("total_revenue"),
    count("*").alias("order_count")
)

#7
w = Window.partitionBy("region") \
          .orderBy(col("total_revenue").desc())

ranked_df = agg_df.withColumn(
    "revenue_rank",
    dense_rank().over(w)
)

#8
ranked_df.cache()
ranked_df.count()

#9
ranked_df.orderBy(
    "region",
    "revenue_rank"
).show()
print("is Cached:",ranked_df.is_cached)
ranked_df.explain()

#10
ranked_df.repartition(2) \
.write \
.mode("overwrite") \
.parquet("ranked_sales")

+------+-----------+-------------+-----------+------------+
|region|   category|total_revenue|order_count|revenue_rank|
+------+-----------+-------------+-----------+------------+
| North|Electronics|   8.551824E8|       6492|           1|
| North|     Office|    4.12114E8|       6492|           2|
| North| Networking| 1.35026775E8|       6494|           3|
| South|Electronics|  2.0242794E9|      19484|           1|
| South|     Office|   6.060058E8|       6495|           2|
| South|Accessories| 1.05018025E8|       6494|           3|
| South| Networking|  8.2684125E7|       6494|           4|
|  West|Electronics|  2.2992404E9|      12985|           1|
|  West|Accessories|    6.42985E7|       6493|           2|
+------+-----------+-------------+-----------+------------+

is Cached: True
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [region#6, category#2300, total_revenue#2304, order_count#2305L, revenue_rank#2326]
      +- InMemoryRelation [region#6, categ